In [2]:
class Node:
    def __init__(self, value, left=None, right=None, color='red'):
        self.value = value
        self.left = left
        self.right = right
        self.color = color   # 'red' или 'black'

    def __repr__(self):
        return f"Node({self.value}, {self.color})"

## Балансировка 
https://habr.com/en/companies/otus/articles/472040/

Вставка рекурсивная: спускаемся вниз, вставляем красный лист, а при возврате вызываем `balance(node)` для каждого предка. Когда мы стоим в узле-**дедушке** `G`, мы видим проблему «красный ребёнок → красный внук» и смотрим на цвет **дяди** `U` (второй ребёнок `G`):

**Случай 1 — дядя красный: перекраска.**
```
      G(black)             G(RED)
     /       \    ->      /      \
  P(red)    U(red)      P(black) U(black)
   |                      |
  X(red)                 X(red)
```
Чёрный цвет «спускается» от дедушки к детям. Теперь сам `G` красный и может конфликтовать со своим родителем — но это починит `balance` на следующем уровне возврата рекурсии.

**Случай 2,3 — дядя чёрный (или его нет): поворот(ы).**
- *Линия* (внук с той же стороны, что и родитель: left-left или right-right) — один поворот вокруг `G`.
- *Излом* (left-right или right-left) — сначала поворот вокруг `P`, чтобы свести к линии, затем поворот вокруг `G`.

После поворота новый корень поддерева красится в **чёрный**, а его дети — в **красный**. Чёрная высота поддерева не меняется, значит выше по дереву ничего чинить не нужно.

In [16]:
class RBtree:
    def __init__(self):
        self.root = None

    
    def is_red(self, node):
        return (node is not None) and (node.color == 'red') #пустой узел - черный

    
    def append(self, value):
        self.root = self._append(self.root, value)
        self.root.color = 'black'          #корень чёрный

    def _append(self, node, value):
        
        if node is None:
            return Node(value, color='red')

        if value > node.value:
            node.right = self._append(node.right, value)
        else:
            node.left = self._append(node.left, value)

        return self.balance(node)

     
    # def find(self, value):
    #     node = self.root
    #     while node:
    #         if value == node.value:
    #             return node
    #         node = node.right if value > node.value else node.left
    #     return None

    
    def height(self, node):
        #максимальная черная высота
        if node is None:
            return 0
        return max(self.height(node.left), self.height(node.right)) + int(node.color == 'black')

    def depth(self, node):
        #обычная высота
        if node is None:
            return 0
        return 1 + max(self.depth(node.left), self.depth(node.right))

     
    def rotation(self, node, direction):
        r"""
        Поворот поддерева с корнем node. Возвращает новый корень поддерева.

        left:                       right:
            node                         new
           /    \                       /    \
          a     new        <->        node     c
               /   \                /   \
              b     c              a     b
        (new поднимается, node опускается; поддерево b меняет родителя)
        Цвета здесь НЕ меняются — этим занимается balance.
        """ 
        if direction == 'left':
            new = node.right
            node.right = new.left
            new.left = node
        elif direction == 'right':
            new = node.left
            node.left = new.right
            new.right = node
        
        return new

    
    def balance(self, node):
        """
        Вызывается для узла node (потенциальный «дедушка») после вставки
        в его поддерево. Ищем красного ребёнка, у которого есть красный ребёнок.
        # Возвращает новый корень этого поддерева.
        """
        #нарушение слева: node.left (красный) -> красный внук (case1 и case2)
        if self.is_red(node.left) and (self.is_red(node.left.left) or self.is_red(node.left.right)):
            if self.is_red(node.right):
                # 1) дядя красный -> перекраска
                node.left.color = 'black'
                node.right.color = 'black'
                node.color = 'red'
            else:
                #2) дядя чёрный -> поворот
                if self.is_red(node.left.right):          #если излом (case 2) -> вращаем до линии -> case 3
                    node.left = self.rotation(node.left, 'left')   
                node = self.rotation(node, 'right') #линия слева, вращаем и перекрашиваем             
                node.color = 'black'
                node.right.color = 'red'

        #зеркальные ситуации
        elif self.is_red(node.right) and (self.is_red(node.right.right) or self.is_red(node.right.left)):
            if self.is_red(node.left):
                # 1) дядя красный -> перекраска
                node.left.color = 'black'
                node.right.color = 'black'
                node.color = 'red'
            else:
                #2) дядя чёрный -> поворот
                if self.is_red(node.right.left):          #если излом (case 2) -> вращаем до линии -> case 3
                    node.right = self.rotation(node.right, 'right')
                node = self.rotation(node, 'left')                #линия справа, вращаем и перекрашиваем
                node.color = 'black'
                node.left.color = 'red'

        return node

    


    def show(self, node=None, indent="", is_root=True):
        #Печать дерева боком: правая ветка сверху
        if is_root:
            node = self.root
        if node is None:
            return
        self.show(node.right, indent + "      ", False)
        print(f"{indent}{node.value}({'R' if node.color == 'red' else 'B'})")
        self.show(node.left, indent + "      ", False)

In [17]:
t = RBtree()
for val in [1, 2, 3, 4, 5, 6, 7]:
    t.append(val)
    print("-------------------------------")
    t.show()

-------------------------------
1(B)
-------------------------------
      2(R)
1(B)
-------------------------------
      3(R)
2(B)
      1(R)
-------------------------------
            4(R)
      3(B)
2(B)
      1(B)
-------------------------------
            5(R)
      4(B)
            3(R)
2(B)
      1(B)
-------------------------------
                  6(R)
            5(B)
      4(R)
            3(B)
2(B)
      1(B)
-------------------------------
                  7(R)
            6(B)
                  5(R)
      4(R)
            3(B)
2(B)
      1(B)


In [14]:
import numpy as np
t = RBtree()
for val in np.random.default_rng().choice(50, size=10, replace=False):
    t.append(val)
    print("-------------------------------")
    t.show()

-------------------------------
6(B)
-------------------------------
      32(R)
6(B)
-------------------------------
      32(R)
7(B)
      6(R)
-------------------------------
      32(B)
            18(R)
7(B)
      6(B)
-------------------------------
            36(R)
      32(B)
            18(R)
7(B)
      6(B)
-------------------------------
            36(B)
      32(R)
                  19(R)
            18(B)
7(B)
      6(B)
-------------------------------
            36(B)
      32(R)
                  22(R)
            19(B)
                  18(R)
7(B)
      6(B)
-------------------------------
            36(B)
      32(R)
                  22(R)
            19(B)
                  18(R)
7(B)
      6(B)
            0(R)
-------------------------------
                  39(R)
            36(B)
      32(R)
                  22(R)
            19(B)
                  18(R)
7(B)
      6(B)
            0(R)
-------------------------------
                  39(R)
            36